# 03 - Model Training

Train Logistic Regression, Ridge, Random Forest, and LightGBM. Build an ensemble. Evaluate on held-out years 2019, 2022, 2023.

In [ ]:

import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data_loader import build_matchup_dataset, build_team_stats
from feature_engineering import prepare_training_data, impute_features, run_rfe
from model import train_and_evaluate, build_ensemble, plot_feature_importance, save_eval_summary, TRAIN_YEAR_MAX


In [ ]:

print("Loading data...")
team_stats = build_team_stats()
matchup_df = build_matchup_dataset(team_stats)
X, y = prepare_training_data(matchup_df)
X = impute_features(X)
years = matchup_df['YEAR'].reset_index(drop=True)

print(f"Total games: {len(X)}")
print(f"Training games (<=2023): {(years <= TRAIN_YEAR_MAX).sum()}")
print(f"Eval games (2019+2022+2023): {years.isin([2019, 2022, 2023]).sum()}")


In [ ]:

# run RFE on training years
train_mask = years <= TRAIN_YEAR_MAX
print("Running RFE...")
selected_features, rfecv = run_rfe(X[train_mask], y[train_mask])
print(f"Selected {len(selected_features)} features")


In [ ]:

# train all models
print("Training models...")
results = train_and_evaluate(X, y, years, selected_features=selected_features)


In [ ]:

# save evaluation summary
summary = save_eval_summary(results)

# print results table
rows = []
for name, res in results.items():
    row = {'model': name, 'cv_logloss': res['cv_logloss']}
    for yr in [2019, 2022, 2023]:
        if yr in res['eval_results']:
            row[f'{yr}_logloss'] = res['eval_results'][yr]['log_loss']
            row[f'{yr}_acc'] = res['eval_results'][yr]['accuracy']
    rows.append(row)
pd.DataFrame(rows).set_index('model')


In [ ]:

# plot random forest feature importance
rf_model = results.get('random_forest', {}).get('model')
if rf_model is not None:
    plot_feature_importance(rf_model, selected_features)
    img_path = os.path.join('..', 'predictions', 'feature_importance_2026.png')
    if os.path.exists(img_path):
        from IPython.display import Image
        Image(img_path)


In [ ]:

# build and evaluate ensemble
train_mask_bool = years <= TRAIN_YEAR_MAX
ens, ens_weight = build_ensemble(results, X[train_mask_bool], y[train_mask_bool],
                                  selected_features=selected_features)
print(f"Ensemble ridge weight: {ens_weight:.3f}")


In [ ]:

# evaluate ensemble on held-out years
from sklearn.metrics import log_loss, accuracy_score

if ens is not None:
    for yr in [2019, 2022, 2023]:
        yr_mask = years == yr
        if yr_mask.sum() == 0:
            continue
        X_eval = X[yr_mask][selected_features].fillna(X[train_mask_bool][selected_features].median())
        y_eval = y[yr_mask]
        probs = ens.predict_proba(X_eval)[:, 1]
        ll = log_loss(y_eval, probs)
        acc = accuracy_score(y_eval, (probs >= 0.5).astype(int))
        print(f"  Ensemble {yr}: log-loss={ll:.4f}, accuracy={acc:.4f}")
